In [0]:
# Cria schemas se ainda não existirem
# Criando schemas no workspace
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.drs_bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.drs_silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.drs_gold")

In [0]:
# importar bibliotecas
import re
import unicodedata
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, lit

In [0]:
# Cria as pastas necessárias dentro do volume

dbutils.fs.mkdirs("/Volumes/workspace/pipeline_estudo/raw_files/input/")
dbutils.fs.mkdirs("/Volumes/workspace/pipeline_estudo/raw_files/checkpoints/bronze_sales/")

In [0]:
#👉 Função para padronizar nomes de colunas

def clean_column_name(col_name):
    col_name = unicodedata.normalize('NFKD', col_name)
    col_name = col_name.encode('ascii', 'ignore').decode('utf-8')
    col_name = col_name.lower()
    col_name = col_name.replace(" ", "_")
    col_name = re.sub(r'[^a-z0-9_]', '', col_name)
    return col_name

In [0]:
# 👉 Lê os arquivos da pasta input com Autoloader

source_path = "/Volumes/workspace/pipeline_estudo/raw_files/input/"

schema = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(source_path)
    .schema
)

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .schema(schema)
    .option("header", True)
    .option("delimiter", ",") \
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .load(source_path)
)

df = df.toDF(*[clean_column_name(c) for c in df.columns])

In [0]:
# Adiciona colunas técnicas da Bronze

df_bronze_v2 = (
    df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit("sales.csv"))
)

In [0]:
# limpa a tabela Bronze para recarga controlada
spark.sql("DROP TABLE IF EXISTS workspace.drs_bronze.sales_v2")

In [0]:
# apaga o checkpoint antigo para reiniciar o controle do stream
dbutils.fs.rm("/Volumes/workspace/pipeline_estudo/raw_files/checkpoints/bronze_sales/", True)
dbutils.fs.mkdirs("/Volumes/workspace/pipeline_estudo/raw_files/checkpoints/bronze_sales/")

In [0]:
# Grava o stream na tabela Bronze final
(df_bronze_v2.writeStream
 .format("delta")
 .option("checkpointLocation", "/Volumes/workspace/pipeline_estudo/raw_files/checkpoints/bronze_sales/")
 .trigger(availableNow=True)
 .toTable("workspace.drs_bronze.sales_v2"))


In [0]:
# Valida a tabela Bronze final
spark.sql("""
SELECT *
FROM workspace.drs_bronze.sales_v2
LIMIT 10
""").display()

In [0]:
# Conta os registros da Bronze
spark.sql("""
SELECT COUNT(*) AS total_bronze_v2
FROM workspace.drs_bronze.sales_v2
""").display()

In [0]:
# Valida duplicidade por chave de negócio

spark.sql("""
SELECT
    id_do_pedido,
    id_do_produto,
    COUNT(*) AS qtd
FROM workspace.drs_bronze.sales_v2
GROUP BY id_do_pedido, id_do_produto
HAVING COUNT(*) > 1
ORDER BY qtd DESC
""").display()